# Pivot Tables and Data Wrangling — Practice Tasks (Task 22)

This practice set is the workout companion to the pivot-table session. Each question hands you a real, messy dataset and a business question, then asks you to bend the data — reshape, pivot, aggregate, parse, and plot — until the answer falls out. The unifying skill is `pivot_table`, but no question can be solved by that one call alone: you will melt, group, extract dates, merge, split text, and use regular expressions along the way, exactly as in the field.

**Q-1** opens with an NFT marketplace dump, large enough that the notebook literally warns you to wait for it to load. You must build a pivot table over `verification_status`, `contract_type`, and `rarity_score` as row/column keys with the total `last_sale_price` as the aggregated value.

**Q-2** is the classic *dirty* car-price dataset. It hides the brand inside a text column (you must derive it), and its `Year` column contains misleading values you must repair before pivoting. The requested output is a brand × year grid holding the **average Price** and the **median Miles** — two different aggregations in one table — finished with a KDE (density) plot of both.

**Q-3** works on India's daily power-generation data: derive a 'required generation' column from `Actual(MU)` and `Excess(+)/Shortfall(-)`, extract the month, find the top-10 most frequent power stations, then pivot station × month and plot it so patterns become visible.

**Q-4 and Q-5** return to text. From a raw `question-answer.csv` you must parse question/answer pairs into a tidy two-column DataFrame where questions are recognised by a `Q<number>` prefix and answers by `Ans<number>`, then filter and print only questions that contain no question mark.

**Q-6 to Q-10** join a `log_file` to an `employee` file on mismatched keys (`EMPLOYEE_ID` vs `emp_id`) to answer activity questions: month-wise activity counts, the top employee per activity, who worked most weekends, which activity dominates business days, and a weekday-wise activity table. Finally **Q-11** repairs the employee data itself: reconstruct missing first names from email + last name, complete the bare email stems with `@campusx.com`, and then show full names of employees starting with 'A' who performed specific activities. Every answer lives in a `# code here` cell.

### ✓ How to use this notebook

Each question ends with a `# code here` cell. Work the plan outlined in its explanation, run your code, and check the printed output against the question's requirements.

In [1]:
import pandas as pd
import numpy as np
import re

pd.options.display.max_columns = None

# Task 1 — NFT Pivot Table

### Q-1 — Total sale price by verification, contract type, and rarity

**Goal:** Build a pivot table using `verification_status`, `contract_type`, `rarity_score`, and `last_sale_price` that shows total values.

### `Question-1:`

The dataset is based on NFTs. This is quite large. So, when you will read the dataset, you have to wait. The link of the dataset: https://tinyurl.com/2pfhncqw

Your task is to make a pivote table by using the columns "verification_status", "contract_type", "rarity_score" and "last_sale_price" to find out the total values.

## 1.1 Approach — choosing keys and the aggregation

The dataset is an NFT marketplace listing; `last_sale_price` is the numeric quantity of interest and the other three columns are categorical or near-categorical descriptors. The task says 'find out the total values', so `aggfunc='sum'` (Pandas' default for `pivot_table` is the **mean**, so you must pass `sum` explicitly).

A clean structure is:
- `index` → one or more of the descriptors (e.g. `verification_status`),
- `columns` → another (e.g. `contract_type`),
- `values='last_sale_price'`,
- `aggfunc='sum'`,
- `fill_value=0` so empty verification/contract combinations read as `0` rather than `NaN`.

Because `rarity_score` is a fine-grained numeric field, stuffing all four columns into a single pivot makes the frame enormous; a better answer often uses two of them on the axes and filters by the third, or pivots once per descriptor and compares. Use `margins=True` to add row/column totals, and inspect the top-level distribution first with `.value_counts()` on `verification_status` and `contract_type` so you know how big the pivot will be. Remember: `pivot_table` is essentially a `groupby` that arranges its result in two dimensions, so `df.groupby([...])['last_sale_price'].sum().unstack()` is a valid cross-check.

Try it in the cell below. Also try `aggfunc='mean'`/`'count'` to see how the picture changes.

In [ ]:
# code here

# Task 2 — Car Prices: Clean, Pivot, Visualise

### Q-2 — Brand × Year average Price and median Miles

**Goal:** For the required brands and years 2018–2022, build a pivot where every row is a brand and each column is an average Price or median Miles for a given year; then plot a KDE chart for Price and Miles.

**Challenges (built into the task):** the brand name is buried inside another column and must be derived; some `Year` values are misleading and must be reset.

### `Question-2:`

You are given a dataset about the cars' price and miles driven of different cars throughout the different years. The link of the dataset: https://tinyurl.com/2r24n45l. Your tasks are
- make a pivot table of the brands (the required brands are given below) from the "Year" 2018 to the year 2022 in which the "Price" is shown as average values and "Miles" are in median values. In this pivote table, every row represents a particular brand and each column represents either average "Price" of a partucular year or median values of "Miles" of a particular year.
- At the end plot a kde chart for the "Price" and "Miles" by using your pivot table.

There are some challenges to solve this task:
- There is no column that is represented to the brand name of the car. You have to find out on your own.
- Some values of "Years" column are misleading. You have to reset this column too that every value should tell a valid meaning.

## 2.1 Approach — engineer the missing keys first

This is the archetypal dirty-data pivot, and the hard work happens *before* `pivot_table`.

**Deriving the brand.** The frame has no brand column, but the model/name text (for example `'2015 Maruti Swift...'`) contains it. Use `.str.split()` on the name column, take the appropriate token (position varies — inspect a few rows first), and assign it to a new `brand` column. Where the brand is the first word, `.str.get(0)` suffices; otherwise regex or a mapping helps.

**Repairing Year.** The complaint that values are 'misleading' means some entries are obviously impossible — e.g. years like `201` or `2105` — caused by truncation or typos. Fix them with `df['Year'] = df['Year'].apply(lambda y: y + 2000 if y < 100 else y)` (or an equivalent rule) and validate with `.value_counts()` / `.sort_values()` on the column.

**The pivot itself.** With two different aggregations requested — average Price *and* median Miles — the cleanest solution is two pivots: `df.pivot_table(index='brand', columns='Year', values='Price', aggfunc='mean')` and `df.pivot_table(index='brand', columns='Year', values='Miles', aggfunc='median')`, restricted to the years 2018–2022 and the named brands. Concatenate them side by side (with `MultiIndex` columns via `pd.concat(..., axis=1, keys=['Price','Miles'])`) if you want the single combined table.

**Visualising.** From the pivot, pipe each sheet into `.plot(kind='kde')` — a kernel-density estimate that shows *the shape of the distribution* of average prices and median miles across the brands/years, rather than raw counts. That is the natural chart when the question is 'how are values distributed?'. Confirm dtypes are numeric first, and `fillna` or `dropna` as appropriate.

In [ ]:
# code here

# Task 3 — India's Daily Power Generation

### Q-3 — Required generation, months, top stations, and a station × month pivot

**Goal:** (a) derive the 'actual generation required' so there is no excess/shortfall; (b) extract and store the month; (c) find the top-10 most frequently appearing power stations; (d) pivot stations against months in calendar order; (e) plot the pivot.

### `Question-3:`

You are given a dataset of **Daily Power Generation in India** of regional wise of all Power Stations. Link of the dataset: https://tinyurl.com/2nq6kugt

Task
- In this dataset, there are many columns. Two of them are `Actual(MU)` and `Excess(+) / Shortfall (-)`. `Actual(MU)` represents the actual power generation. `Excess(+) / Shortfall (-)` tells is that generated power is excess or shortfall for that particular day of a power station. You have to find out what should be the actual power generation required for that day by the power stations that there would not be any excess or shorfall power. For the result, make a new column.
- Find out the month of the day and store it as a new column.
- Find out top 10 frequently appeared power stations in this dataframe.
- Next create a pivot table of which every row represents a power station and every column represents a month in a order. Like January, February, March...
- It is hard to get inside if you look through the only pivot table. So plot the pivot table.

## 3.1 Approach — arithmetic columns, date parsing, ranking, and reshaping

**Required generation.** If `Actual(MU)` is what was produced and `Excess(+)/Shortfall(-)` is the gap, then the requirement is `Actual - Excess/Shortfall` for an excess day (you must *reduce* by the surplus) and `Actual + Shortfall` on a shortfall day. In one expression, `required = df['Actual(MU)'] - df['Excess(+) / Shortfall (-)']`, because a negative shortfall subtracts as an addition. Sanity-check a handful of rows: an excess row (`+5`) should yield a required value below the actual; a shortfall row (`-5`) should yield one above.

**Month extraction.** Parse the date column with `pd.to_datetime(...)` and pull `df['date'].dt.month_name()` into a new column. This gives strings in calendar order once sorted, and is the key needed for the pivot's columns.

**Top 10 stations.** `df['Power Station'].value_counts().head(10)` ranks them by number of appearances; `.index` gives the names.

**The pivot.** `df.pivot_table(index='Power Station', columns='month', values=<a numeric column>, aggfunc='mean'/'sum')`. To get monotonic January→December ordering, build the column order explicitly with `pd.Categorical` on the month name (categories set to the 12 month names) before pivoting, or reindex the resulting columns to a month list.

**Plotting.** `pivot.plot()` gives a line per station across the months; `kind='bar'` or a heatmap variant is often clearer given many stations. The stated motive — 'it is hard to get inside if you only look at the pivot table' — is the general rule: aggregate first, *then* visualise so the eye can find the seasonal/regional patterns in a wide grid. Finally, restrict the plot to the top stations so the chart is legible.

In [ ]:
# code here

# Task 4 — Turning a Text File into a Q&A DataFrame

### Q-4 — Parse `question-answer.csv`

**Goal:** Build a two-column DataFrame, `question` and `answers`, where questions begin with `Q<number>` (e.g. `Q1`) and answers begin with `Ans<number>` (e.g. `Ans1`). Watch the column names carefully.

###`Q-4` You are given a file `question-answer.csv`. Your task is to make a dataframe from it with two columns - `question` and `answers`.

* Questions in the file start from `Q<number>`; E.g.  `Q1` denotes question no. `1`
* Answers starts from `Ans<number>`; Eg. `Ans1` denotes answer of question no `1`
* MAke sure you look at columns name carefully

CSV File - "https://drive.google.com/file/d/10rmV3XrVtzpDTtYZF3UtCdcU0ajBJjGY/view?usp=share_link"

## 4.1 Approach — read it, split it, stitch it

The CSV is a text dump rather than a tidy table, so read it loosely (`pd.read_csv(..., header=None)` or the appropriate separator), inspect its actual shape, and treat each non-empty line as a token. The pattern is regular:

1. Read the file and get all lines as a Series/array (drop blanks — `dropna` after converting blanks to `NaN`, or filter `.str.strip() != ''`).
2. Questions are the lines that **start with the letter Q followed by a digit** — select with `.str.match(r'^Q\d+')`. Answers start with **`Ans` plus a digit** — `.str.match(r'^Ans\d+')`.
3. Strip the marker off each: `q.str.replace(r'^Q\d+\s*', '', regex=True)` and the analogous removal for `Ans`. Multi-line questions/answers are the tricky part — decide whether a continuation line (one that starts with neither marker) belongs to the previous token, and if so glue it on.
4. Assemble the two cleaned arrays into `pd.DataFrame({'question': q_list, 'answers': a_list})`.

Key traps called out by the task: **column names** must be exactly `question` and `answers` (singular answer? the task says `answers`), and the marker removal must leave no leading `Q1` text behind. Preview with `.head()` — if every question still shows a `Q<number>` prefix your regex did not match, and if row counts differ between questions and answers you lost or gained a continuation line. Merge/validate alignment of the `n` in `Qn` with `Ansn` before trusting the frame.

In [ ]:
# code here

# Task 5 — Filter Questions Without a Question Mark

### Q-5 — Print Q&A for questions lacking `?`

**Goal:** From the DataFrame built in Q-4, print the question and answer of those entries whose **question text contains no question mark (`?`)**.

###`Q-5`: Print Question and answer of those questions which does not contains any question mark (`?`).


## 5.1 Approach — a boolean mask on text

The tool is `.str.contains('?')` combined with negation. Because `?` is a regular-expression metacharacter, passing it literally to `contains` may surprise you — in a regex `?` means 'the previous token, optionally', so the safest habit is `regex=False`:

`mask = df['question'].str.contains('?', regex=False)`

Then `df[~mask]` selects the rows where the question has **no** question mark, and printing `question` + `answers` for those rows gives the requested output. Equivalent formulations: `df[df['question'].str.find('?') == -1]`, or escaping the metacharacter in a regex (`r'\?'`, which is what you would use if the pattern needed to be a regex anyway).

The deeper point is that 'does this text contain a given substring?' should never be answered with a Python `for` loop and `in`. Vectorised `.str` methods handle the whole column at once, tolerate missing values, and compose directly into DataFrame indexing. After this you have the complete 'parse text into a table, then filter on its content' pipeline that underlies search boxes, FAQ tooling, and log mining.

In [ ]:
# code here

# Tasks 6–10 — LOG and EMPLOYEE

### Q6–Q10 — Activity analytics across two files

**Goal:** Using `log_file` and `employee` (keys differ: `EMPLOYEE_ID` vs `emp_id`), answer: month-wise activity counts; the employee with the most activity in January; the employee who worked most weekends; the activity logged most on business days; and a weekday-wise activity table.

###`Q 6-10` LOG and EMPLOYEE 
6. Show `activity` details month wise. Show count for each `activity` 
7. Find employee who did most `activity` in January month.
8. Employee who have worked most no of times on Weekends.
9. Which activity is logged most on buisness days.
10. Week Days wise activity table.
```
log_file = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT-OMrmmNdOTM-B5f5F1EpCutMVG230UZiLvqlsg0NIKUKR3yrqiI2r1pEX-LvSEk-3WwySPYtvbBC-/pub?gid=1937029224&single=true&output=csv")
employee = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQ5AuFqRjSZVBKm5zyDxquX6utubq1DJKkYDI70vjeidAnyAu70KMSYpMYzeVSNVTeUIJBpfF6jU5E6/pub?gid=798824749&single=true&output=csv")

```

Note(for common field):- Employee file has `EMPLOYEE_ID` and LOG file has `emp_id`

## 6.1 Approach — join first, engineer dates, then aggregate

**Join.** The employee and log files share identity under different names. Normalise by renaming (`log_file.rename(columns={'emp_id':'EMPLOYEE_ID'})`) or by merging with `left_on='emp_id', right_on='EMPLOYEE_ID'`, then keep only the columns you need. After the merge, each log row carries the employee's name — required for several of the questions.

**Dates and flags.** Parse the log's date column with `pd.to_datetime`. From it derive the month (`dt.month_name()`), the weekday name (`dt.day_name()`), and a weekend flag (`dt.dayofweek >= 5`, since Monday=0). These three derived columns power every question.

**Q6** — month-wise activity: `log.groupby(['month', 'activity']).size()` or a `crosstab(month, activity)` gives counts per activity per month.

**Q7** — most active in January: filter `month == 'January'`, then `groupby('name').size().idxmax()`.

**Q8** — most weekend work: filter the weekend flag to `True`, then `groupby('name').size().idxmax()`.

**Q9** — top activity on business days: filter `dayofweek < 5` (or `~weekend`), then `groupby('activity').size().idxmax()`.

**Q10** — weekday-wise table: `pd.crosstab(log['day_name'], log['activity'])`, ideally reordered to Monday→Sunday so the reading follows the week.

**Watch-outs:** duplicates in the employee file can explode row counts after the merge — `drop_duplicates` on the key first; and 'most' questions need the `.size()`/`.count()` aggregation before `idxmax()`, not a raw sort of the whole frame.

In [ ]:
# code here

# Task 11 — Cleaning the Employee File

### Q-11 — Repair names, complete emails, filter by activity

**Goal:** (1) Fill missing first names using the email stem and last name; (2) append `@CAMPUSX.COM` to the bare email stems; (3) show full names of employees whose name starts with 'A' and who performed any of `['Incpection', 'Cleaning', 'Checking']`.

### `Q-11`:

1. There are missing value in first name of employee. Fill it using email and last name field.

> E.g- `email` -> JMURMAN and `last_name` -> 'Urman' -> so make it's first name as 'JM', Sort of (Email- lastname). 

> Email is constructed from initials of first name concate with lastname. Your Task is to fill first name initials in missing data.

2. You can see email field don't have any domain name. Change this to full email addreess with domain as 'campusx.com'.
E.g- Email field -> `JMURMAN`  result -> `JMURMAN@CAMPUSX.COM`

3. Show Full Name of all the employees whose name starts with 'A' and has done any of these activity ['Incpection', 'Cleaning', 'Checking]

## 7.1 Approach — regex extraction, string concatenation, and multi-condition filtering

**Part 1 — reconstruct missing first names.** The rule is given: the email stem is the *initials of the first name concatenated with the last name*. So for `email = 'JMURMAN'` and `last_name = 'Urman'`, the first name initials are everything before the last name's characters, i.e. `'JM'` → first name `'JM'` (or expanded from there). Programmatically: for each row with a null first name, strip the `last_name.upper()` suffix off the email (`email.replace(last_name.upper(), '')`), giving the initials, and assign it as the first name. Vectorise with `.str.replace` on the upper-cased last name, or apply row-wise where names vary in case.

**Part 2 — complete the domain.** `df['email'] = df['email'].str.upper() + '@CAMPUSX.COM'`. Guard against rows that already have a domain (a `~df['email'].str.contains('@')` mask) so you never double-append.

**Part 3 — filtered full names.** Build `full_name = first_name + ' ' + last_name` (handle the blanks you just filled). 'Starts with A' is `full_name.str.startswith('A')` (or `str.upper().str.startswith('A')` to be case-insensitive). The activity condition is a membership test: `log['activity'].isin(['Incpection', 'Cleaning', 'Checking'])` — note the dataset's own typo `'Incpection'`, which you must match exactly. Combine both masks with `&` after aligning the join so each employee's activities and name sit on the same rows, then `print(full_name[mask])`.

The general lesson: missing data is often *recoverable by rule* — when one column is a deterministic function of others (`initials(lastname)`), you can rebuild it instead of dropping rows. And `isin` is the correct, readable way to test against a list of allowed values.

## Summary

Task 22 was a pivot-table gauntlet wrapped in real-world messiness. Q-1 pivoted NFT sales by verification status, contract type, and rarity with a `sum` aggregation. Q-2 forced the discipline of *cleaning before shaping* — deriving a hidden `brand` column and repairing corrupted `Year` values — then produced a brand × year grid mixing **mean Price** and **median Miles** and visualised the distributions with KDE plots. Q-3 engineered a required-generation column from `Actual(MU)` and `Excess/Shortfall`, mined months, ranked the top-10 power stations, pivoted station × month, and plotted to expose patterns. Q-4 and Q-5 translated a text file into a tidy `question`/`answers` DataFrame using regex-prefixed parsing, then filtered with `.str.contains('?', regex=False)`. Q6–Q10 joined log and employee files across mismatched keys and answered month-, weekend-, business-day-, and weekday-level activity questions via `groupby`, `crosstab`, and `idxmax`. Q-11 restored missing names for rule-recoverable columns, completed email domains, and combined `startswith` with `isin` for a two-condition filter. The recurring theme: shape, clean, derive, aggregate, visualise — in that order.

In [ ]:
# code here